# ForenSight — AI-Image Detector (Phase 2: realistic images)

Same idea as Phase 1, but on a **realistic, higher-resolution** dataset
(`Parveshiiii/AI-vs-Real` on Hugging Face) at **224x224** — so the model works
on real photos, not just tiny 32x32 ones.

Runtime -> Change runtime type -> **GPU**, then run the cells top to bottom.


## 1. Check the GPU is on


In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Download the dataset (Hugging Face — no key needed)
This dataset is public, so no login/token is required. It downloads into
`aivsreal/` with `train/` and `test/` folders, each holding `ai/` and `real/`.


In [ ]:
!pip install -q -U huggingface_hub
from huggingface_hub import snapshot_download
snapshot_download(repo_id='Parveshiiii/AI-vs-Real', repo_type='dataset',
                  local_dir='aivsreal')
print('Folder structure:')
!ls aivsreal && echo '---' && ls aivsreal/train

## 3. Load the images (224x224)
If the folder names printed above are different from `train`/`test` or `ai`/`real`,
tell me and I'll adjust these two paths.


In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32
train_ds = tf.keras.utils.image_dataset_from_directory(
    'aivsreal/train', image_size=IMG_SIZE, batch_size=BATCH,
    label_mode='binary', shuffle=True, seed=42)
test_ds = tf.keras.utils.image_dataset_from_directory(
    'aivsreal/test', image_size=IMG_SIZE, batch_size=BATCH,
    label_mode='binary', shuffle=False)
class_names = train_ds.class_names
print('Classes (index order):', class_names)   # expect ['ai', 'real'] -> ai=0, real=1
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 4. Peek at a few images


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(labels[i][0])])
        plt.axis('off')
plt.show()

## 5. Build the model (transfer learning, MobileNetV2 @ 224)


In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = layers.RandomFlip('horizontal')(inputs)
x = layers.RandomRotation(0.05)(x)
x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)   # P(REAL)
model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Train (frozen backbone)


In [ ]:
history = model.fit(train_ds, validation_data=test_ds, epochs=5)

## 7. Fine-tune (unfreeze top layers for a boost)
This usually adds a few % accuracy on realistic data — worth running here.


In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy', metrics=['accuracy'])
history2 = model.fit(train_ds, validation_data=test_ds, epochs=5)

## 8. Evaluate — precision, recall, confusion matrix


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
y_true, y_prob = [], []
for images, labels in test_ds:
    y_true.extend(labels.numpy().flatten().tolist())
    y_prob.extend(model.predict(images, verbose=0).flatten().tolist())
y_true = np.array(y_true); y_pred = (np.array(y_prob) >= 0.5).astype(int)
print('Order:', class_names)
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## 9. Save + download the model
Saves `ai_detector_v2.keras` and downloads it. **Rename it to `ai_detector.keras`** and
put it in `forensight/models/` (replacing the CIFAKE one). ForenSight auto-detects the
224 input size, so no code change is needed.


In [ ]:
model.save('ai_detector_v2.keras')
from google.colab import files
files.download('ai_detector_v2.keras')

## 10. Try it on your own image
P(AI) = 1 - P(real).


In [ ]:
from google.colab import files
up = files.upload()
for fn in up:
    img = tf.keras.utils.load_img(fn, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)[None, ...]
    p_real = float(model.predict(arr, verbose=0)[0][0])
    print(f'{fn}:  P(AI) = {(1 - p_real):.1%}   P(real) = {p_real:.1%}')

---
### After this
1. Download `ai_detector_v2.keras`, rename to **`ai_detector.keras`**, drop it in
   `forensight/models/` (replace the old one).
2. Restart ForenSight -> Run / Refresh Analysis -> the ML line uses the new model.
3. Test on your `gambar/` images and report precision/recall vs the rule-based detector.
